# 04 · E1 baseline: untrained Qwen3-VL-4B on the full test set (Step 2.3)

**Runtime: L4 GPU** (Runtime → Change runtime type → **L4 GPU**).

Scores the off-the-shelf model on every test example: about 2,900 answering, 4,300 answerability and 7,750 quality examples. Same pipeline as the T4 check (notebook 03).

Expect about **45–75 minutes**. Keep the tab open. Results are saved to Drive after each task, so a disconnect only loses the task in progress (see **If Colab disconnects** at the end).

Every command runs through `sh()`, which stops with an error if the command fails.

## 1 · Settings, Drive, code

In [ ]:
import os
import subprocess

from google.colab import drive

REPO_URL = "https://github.com/Vishwapujara/vizwiz-assistant.git"
BRANCH = "main"
REPO_DIR = "/content/vizwiz-assistant"
DRIVE_ROOT = "/content/drive/MyDrive/vizwiz"
DATA = f"{DRIVE_ROOT}/datasets"
LOCAL = "/content/data"
RUNS = f"{DRIVE_ROOT}/runs"
VENV = "/content/venv"
PY = f"{VENV}/bin/python"
RUN_ID = "e1-baseline"
TASKS = "answer answerability quality"  # if Colab disconnects: only the unfinished ones


def sh(cmd, **env):
    """Run a shell command, stream its output, and raise if it fails."""
    proc = subprocess.Popen(
        cmd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env={**os.environ, **env},
    )
    for line in proc.stdout:
        print(line, end="")
    if proc.wait():
        raise RuntimeError(
            f"Command failed (exit {proc.returncode}). Stop here and paste the output above."
        )


drive.mount("/content/drive")
sh(
    f"if [ -d {REPO_DIR}/.git ]; then git -C {REPO_DIR} fetch -q origin {BRANCH} && git -C {REPO_DIR} reset -q --hard origin/{BRANCH}; else git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}; fi"
)
sh(f"git -C {REPO_DIR} log --oneline -1")
sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")

**Paste back:** the commit line and the GPU line (should say NVIDIA L4).

## 2 · Pinned environment (~3–5 min)

In [ ]:
%%time
sh("pip install -q uv")
sh(f"uv venv -q --clear --python 3.13 {VENV}")
sh(
    f"uv pip install -q --python {PY} -e {REPO_DIR} -r {REPO_DIR}/requirements/colab-gpu.txt -c {REPO_DIR}/requirements/colab-gpu.lock.txt"
)
sh(f"{PY} -m vizwiz.compat versions")

## 3 · Check label files, unzip test photos (~1 min)
The test split is the official val set, so only `val.zip` is needed.

In [ ]:
import hashlib
import json

expected = json.load(open(f"{REPO_DIR}/results/splits/files.json"))
for name in ("vqa_test.jsonl", "quality_test.jsonl"):
    digest = hashlib.sha256(open(f"{DATA}/processed/{name}", "rb").read()).hexdigest()
    assert digest == expected[name]["sha256"], f"{name} differs from the committed hash"
    print(f"{name}: matches committed hash")
sh(f"{PY} -m vizwiz.data.download extract --root {DATA} --dest {LOCAL} --splits val")

## 4 · Run the baseline (~45–75 min)
Downloads the model (~9 GB) to local disk, starts vLLM, then runs every test example of each task. Progress lines print per batch of 64.

In [ ]:
%%time
sh(
    f"{PY} -m vizwiz.eval.run_eval --records {DATA}/processed --images {LOCAL}/images --split test --tasks {TASKS} --run-id {RUN_ID} --out {RUNS}/{RUN_ID}",
    HF_HOME="/content/hf_cache",
)

**Paste back:** the three lines that start with `[answer] vqa_accuracy`, `[answerability] average_precision`, `[quality] macro_f1`, and any warning or error.

## 5 · Answerability reply formats (safe to paste)
Counts only, no text: how many replies were valid, and what kind the invalid ones were.

In [ ]:
m = json.load(open(f"{RUNS}/{RUN_ID}/metrics.json"))["tasks"]["answerability"]["metrics"]
print("invalid %:", m["invalid_output_pct"], " kinds:", m["invalid_output_kinds"])
print("confusion:", m["confusion"])

**Paste back:** both lines.

## 6 · Flush Drive, then disconnect
Upload back from `MyDrive/vizwiz/runs/e1-baseline/`: **`metrics.json`** only (summary numbers; the prediction files stay on Drive).

Then Runtime → Disconnect and delete runtime.

### If Colab disconnects
Reconnect (L4), run cells 1–3, then in cell 1 change `TASKS` to only the tasks whose `[task] ...` result line had **not** printed (for example `TASKS = "quality"`), rerun cell 1, and run cells 4–6. Finished tasks are kept in `metrics.json`.

In [ ]:
drive.flush_and_unmount()
print("Drive flushed. Safe to disconnect.")